In [2]:
#Refer: 0.1.2-Langchain_simple_exmpl.ipynb for setup.
#!pip install torch==2.2.2
#!pip install transformers==4.37.2
#if any issues with older installation, uninstall and reinstall approprite version
#!pip uninstall transformers torch huggingface-hub tokenizers -y
#!pip uninstall transformers -y
#!pip install torch==2.2.2
#!pip install transformers==4.37.2

In [1]:
import transformers
print(transformers.__version__)
print(transformers.__file__)
import os
import transformers.models.t5
print(transformers.models.t5.__file__)

4.55.2
E:\Lesson_2_demos\venv\lib\site-packages\transformers\__init__.py
E:\Lesson_2_demos\venv\lib\site-packages\transformers\models\t5\__init__.py


In [2]:
from transformers import pipeline
pipe = pipeline(
    "text2text-generation", 
    model="google/flan-t5-large")


Device set to use cpu


In [8]:
prompt = "Write a catchy tagline for a airline."
from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)
response = llm.invoke(prompt)

In [9]:
print(response), print(type(response))

i want to fly with you
<class 'str'>


(None, None)

In [10]:
print(pipe(prompt))

[{'generated_text': 'i want to fly with you'}]


In [11]:
print(pipe(prompt)[0]["generated_text"])

i want to fly with you


In [ ]:
#Function Approach
from transformers import pipeline
prompt = "Write a catchy tagline for a airline."
pipe = pipeline("text2text-generation", model="google/flan-t5-large")
def get_completion(prompt):
    response = pipe(prompt,max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())

get_completion(prompt)

Device set to use cpu


'i want to fly with you'

In [ ]:
#Looking at different approaches
prompt = "Write a catchy tagline for a coffee brand."
#Uncomment below to test
'''
#Option 1
from transformers import pipeline
pipe = pipeline("text2text-generation", model="google/flan-t5-large")
print(pipe(prompt)[0]["generated_text"])

'''
#Uncomment below to test
'''
#Option 2
#Earlier we did this -- function approach
pipe = pipeline("text2text-generation", model="google/flan-t5-large")
def get_completion(prompt):
    response = pipe(prompt,max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())

get_completion(prompt)

'''

#Option 3 i.e. using chain
#Earlier we also did this -- chain approach
#prompt = "Write a catchy tagline for a coffee brand." 
#--prompt is just a plain Python string. 
#--In LangChain, the | operator expects a Runnable on the left side, not a raw string. A string can’t be piped directly into an LLM.
#so we have to use prompt template
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
# Load the model and tokenizer locally
model_name = "google/flan-t5-large"  # You can also use "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)
llm = HuggingFacePipeline(pipeline=pipe)

#with variables
# Create a prompt template
prompt = PromptTemplate(
    input_variables=["product"],
    template="Write a catchy tagline for a {product}."
)

chain = prompt | llm

# Invoke chain
response = chain.invoke({"product": "coffee brand"})
print(response)

'''
#or using without variables
prompt = PromptTemplate.from_template(
    "Write a catchy tagline for a coffee brand."
)

chain = prompt | llm
response = chain.invoke({})
print(response)
'''


In [6]:
brand_name = input("Enter the brand name: ")
product_name = input("Enter the product name: ")
product_category = input("Enter the product category (e.g., drink, oil, cream, bar): ")
product_description = input("Enter a short description of the product: ")
key_features = input("List the key product features (comma-separated): ")
revenue_target = input("Enter the target revenue increase (e.g., '24%'): ")
target_age_range = input("Enter the target age range (e.g., '22–55 years'): ")
target_interests = input("Enter target audience interests (comma-separated): ")
target_pain_points = input("Enter key audience pain points: ")
campaign_focus = input("Enter the campaign focus (e.g., 'Brand Awareness', 'Pre-orders', 'Seasonal Launch'): ")



Enter the brand name:  Nutri Bely
Enter the product name:  Protein drink
Enter the product category (e.g., drink, oil, cream, bar):  Health drnk
Enter a short description of the product:  Drink which is not just for thirst rather for strength
List the key product features (comma-separated):  High protein, low sugar
Enter the target revenue increase (e.g., '24%'):  30%
Enter the target age range (e.g., '22–55 years'):  20-50
Enter target audience interests (comma-separated):  health, looks and fitness
Enter key audience pain points:  most of products in market are with high sugar, low protein
Enter the campaign focus (e.g., 'Brand Awareness', 'Pre-orders', 'Seasonal Launch'):  brand awareness


In [7]:
# Construct the dynamic prompt
prompt = f"""
You are a senior marketing strategist with 25+ years of experience in the wellness, fitness, and healthy lifestyle industry.

Create a comprehensive product launch campaign for a new product.

**Context:**
{brand_name} aims to strengthen its position in the wellness and fitness market and achieve at least a {revenue_target} revenue increase compared to the previous year.
The campaign should focus on brand differentiation, audience engagement, and conversion.

**Product Details:**
- Product Name: {product_name}
- Product Category: {product_category}
- Description: {product_description}
- Key Features: {key_features}

**Target Audience:**
- Age Range: {target_age_range}
- Interests: {target_interests}
- Pain Points: {target_pain_points}

**Competitor Context:**
Analyze 3 leading competitors in the same product category from major platforms (e.g., Amazon, health-food retailers, or direct-to-consumer brands).
Highlight:
- Strengths
- Weaknesses
- Opportunities for {brand_name} to differentiate.

**Campaign Focus:** {campaign_focus}

**Deliverables:**
1. Campaign strategy summary (positioning, message, and insights)
2. Tagline
3. Instagram post (visual concept + caption)
4. Call-to-action (for pre-orders or early adoption, including any limited-time offer)
5. Differentiation strategy and growth recommendations to meet the revenue goal.

**Tone & Style:** Confident, aspirational, and aligned with modern wellness branding — a balance of science-backed credibility and lifestyle inspiration.
"""

In [12]:
prompt

'\nYou are a senior marketing strategist with 25+ years of experience in the wellness, fitness, and healthy lifestyle industry.\n\nCreate a comprehensive product launch campaign for a new product.\n\n**Context:**\nNutri Bely aims to strengthen its position in the wellness and fitness market and achieve at least a 30% revenue increase compared to the previous year.\nThe campaign should focus on brand differentiation, audience engagement, and conversion.\n\n**Product Details:**\n- Product Name: Protein drink\n- Product Category: Health drnk\n- Description: Drink which is not just for thirst rather for strength\n- Key Features: High protein, low sugar\n\n**Target Audience:**\n- Age Range: 20-50\n- Interests: health, looks and fitness\n- Pain Points: most of products in market are with high sugar, low protein\n\n**Competitor Context:**\nAnalyze 3 leading competitors in the same product category from major platforms (e.g., Amazon, health-food retailers, or direct-to-consumer brands).\nHighl

In [5]:
print(pipe(prompt)[0]["generated_text"])

NUtri dely protein choco bars are high in portein, low in sugar, and low in carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NUtri dely protein bars are high in portein, low in sugar, and low carbs. NU


### Switching to GPT models and Azure/OpenAI as flan-t5 models wouldnt be a better choice for generation as needed.

In [13]:
#Switching to GPT models and Azure/OpenAI
#OpenAI SDK (openai.AzureOpenAI)
import os
import dotenv
import openai
from openai import AzureOpenAI
from dotenv import load_dotenv

#load_dotenv("/content/.env")
load_dotenv()

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

deployment_name = os.getenv("AZURE_DEPLOYMENT_NAME")

In [17]:
response = client.chat.completions.create(
    messages=[{"role": "user", "content": prompt}],
    max_completion_tokens=400,
    temperature=1.0,
    top_p=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    model=deployment_name
)

#print("type:", type(response)) , 
#print("response:", response),
print(response.choices[0].message.content)


Certainly! Here’s a comprehensive product launch campaign for Nutri Bely’s new high-protein, low-sugar protein drink.

---

### 1. **Campaign Strategy Summary**

**Positioning:**  
Nutri Bely Protein Drink is the definitive answer for consumers demanding both *strength* and *smart nutrition*—a drink formulated to fuel active lifestyles, cut unnecessary sugar, and elevate everyday energy. Positioned as more than just hydration, it’s *strength in a bottle*.

**Core Message:**  
- “Drink for strength, not just thirst.”  
- Science-backed protein with minimal sugar—ideal for active, health-conscious adults.

**Insights:**  
- Target market is frustrated with “healthy” drinks overloaded with sugar and lacking real protein.
- Consumers crave convenient solutions that don’t sacrifice genuine fitness benefits for taste or trendy branding.

---

### 2. **Tagline**

**“Sip Strong. Live Bold.”**

---

### 3. **Instagram Post**

#### **Visual Concept:**
- **Background:** Cool, minimalist gym setti

In [18]:
def get_completion(prompt, deployment_name=deployment_name):
    """
    Get a chat completion from Azure OpenAI.
    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.
    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )

        return response.model_dump()  # Return the full response as dict
        #return response.choices[0].message.content

    except Exception as e:
        return {"error": str(e)}

In [19]:
get_completion(prompt)

{'id': 'chatcmpl-DlbtopsiGpZx4YIqoGUac37Mej0ei',
 'choices': [{'finish_reason': 'length',
   'index': 0,
   'logprobs': None,
   'message': {'content': 'Certainly! Here’s a comprehensive product launch campaign for Nutri Bely’s new protein drink:\n\n---\n\n### 1. **Campaign Strategy Summary**\n\n**Positioning:**  \nNutri Bely’s Protein Drink is the smart choice for modern wellness seekers who demand strength, not just hydration. Unlike sugary, low-protein alternatives, Nutri Bely delivers high-quality protein with minimal sugar, supporting active lifestyles and fitness goals.\n\n**Core Message:**  \n“Drink for Strength, Not Just Thirst.”  \nNutri Bely empowers you to fuel your body with what it truly needs—clean, potent protein—so you can look, feel, and perform your best.\n\n**Key Insights:**  \n- Consumers are increasingly aware of hidden sugars in “healthy” drinks.\n- There’s a gap for a protein drink that’s genuinely low in sugar and high in protein.\n- Wellness buyers want science

In [20]:
#Or
response = get_completion(prompt)

In [ ]:
print(response['choices'][0]['message']['content'])

In [22]:
#If using langchain & AzureOpenAI
from langchain_openai import AzureOpenAI
# Initialize client once
client_lc = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name="gpt-4.1", # chat completions would work with this model
    temperature=0.5,
    top_p=0.8,
    max_tokens=512
)

In [23]:
def get_completion_lang_azure(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response
    except Exception as e:
        return {"error": str(e)}

In [24]:
response_lc = get_completion_lang_azure(prompt)

In [25]:
print(response_lc)

{'error': "Error code: 400 - {'error': {'code': 'OperationNotSupported', 'message': 'The completion operation does not work with the specified model, gpt-4.1. Please choose different model and try again. You can learn more about which models can be used with each operation here: https://go.microsoft.com/fwlink/?linkid=2197993.'}}"}


In [ ]:
#We can use different newer model to test with AzureOpenAI from langchain or proceed with AzureChatOpenAI

In [26]:
#Use LangChain Chat model , If you want messages-style chat
from langchain_openai import AzureChatOpenAI

client_lc = AzureChatOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name="gpt-4.1",
    temperature=0.5,
    max_tokens=512
)

In [27]:
def get_completion_lang_azure(prompt):
    try:
        response = client_lc.invoke(prompt)
        return response.content
    except Exception as e:
        return {"error": str(e)}

In [28]:
response_lc = get_completion_lang_azure(prompt)

In [ ]:
print(response_lc)

In [ ]:
#Writing activity example
# Construct the dynamic prompt
prompt = f"""Act as a senior direct-response copywriter.

Write a 500-word blog post announcing a new eco-friendly reusable water bottle.

Target audience:
- Environmentally conscious consumers
- Fitness enthusiasts
- Health-conscious professionals

Goals:
- Increase product awareness
- Differentiate from competing reusable bottles
- Encourage purchases

Use the following structure:

1. Attention-grabbing headline
2. Introduce the problem of single-use plastic bottles
3. Explain why many reusable bottles still have drawbacks
4. Introduce our product as a better alternative
5. Highlight the following benefits:
   - Sustainable materials
   - Durable construction
   - Safe for everyday hydration
   - Lightweight and convenient
6. Explain environmental and health benefits
7. End with a strong call to action

Writing guidelines:
- Use short paragraphs
- Focus on benefits over features
- Avoid hype and exaggerated claims
- Sound trustworthy and human
- Write at an 8th-grade reading level

**Tone & Style:** Confident, aspirational, and aligned with our goals.
"""

In [35]:
response = client_lc.invoke(prompt)

In [36]:
print(response.content)

**Meet the Water Bottle That’s Changing the Game—for You and the Planet**

Tired of seeing plastic bottles pile up in landfills and oceans? Many reusable bottles promise a greener future, but they’re often heavy, leaky, or made with questionable materials.

Introducing the EcoFlow Bottle—designed for those who care about their health and the Earth. Made from sustainable materials, it’s tough enough for any workout, safe for daily use, and so light you’ll forget it’s in your bag.

Switching to EcoFlow means less plastic waste and cleaner hydration. Ready to make a difference? Choose EcoFlow today and sip smarter—for yourself and the planet.


In [ ]:
#When using openAI we need to use ChatOpenAI
import os

from langchain_openai import ChatOpenAI

openai_base_url = os.getenv("OPENAI_BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-3.5-turbo"

llm = ChatOpenAI(
    base_url=openai_base_url,
    api_key=openai_api_key,
    model=model,
    temperature=0.2,
    max_tokens=700,
)

#then use llm defined for chaining or within function or llm.invoke